In [ ]:
import numpy as np

# user input
sz = [0.01, 0.01, 0.01]   # total size of the network in m
shape = [5, 5, 5]         # number of pores in x, y and z direction
mu = 1e-3                 # fluid viscosity in Pa s

def get_throat_diameter(coords, conns):
    d_mean = 1e-3
    sigma = 1e-4
    d_throat = np.random.normal(loc=d_mean, scale=sigma, size=conns.shape[0])  # Gaussian distribution
    return np.clip(d_throat, 1e-6*d_mean, None)  # avoid negative values
    # return np.full(conns.shape[0], fill_value=d_mean)  # all throats have the same diameter


In [ ]:
# generate network
print(f"Generating Network or shape ({shape}) and size ({sz})... ", end="")
dx = [s/n for s,n in zip(sz, shape)]
coords = [(np.arange(nx)+0.5) * d for nx, d in zip(shape, dx)]
coords = np.meshgrid(*coords, indexing="ij")
coords = np.column_stack([c.ravel() for c in coords])
pore_inlet = np.where(coords[:, 0] < dx[0])[0]
pore_outlet = np.where(coords[:, 0] > (sz[0]-dx[0]))[0]

idx = np.arange(np.prod(shape)).reshape(*shape)
cx = np.column_stack([ idx[:-1, :, :].ravel(), idx[1:, :, :].ravel()])
cy = np.column_stack([ idx[:, :-1, :].ravel(), idx[:, 1:, :].ravel()])
cz = np.column_stack([ idx[:, :, :-1].ravel(), idx[:, :, 1:].ravel()])
conns = np.vstack([cx, cy, cz])
print("done")

# compute throat conductivity
print("Computing throat conductivity... ")
r_throat = (get_throat_diameter(coords=coords, conns=conns) * 0.5)
l_throat = np.sqrt(np.sum((coords[conns[:, 1], :] - coords[conns[:, 0], :])**2, axis=1))
g = (r_throat**4 * np.pi /(8 * mu * l_throat)).reshape(-1, 1)
print("done")

# let's do some really basic Jacobi iterations
# a good initial guess does help convergence
# Note, that this is NOT ideal for real modeling and just a very simple implementation
# for this example. If you are interested in better methods, have a look at OpenPNM (openpnm.org)
# or PNM-ICE (https://github.com/multiscale-operations-in-porous-systems/pnm-ice)
n_iter = 50               # number of Jacobi iterations
n_out = 10                # output intervals
print("\nCompute flow rates")
print(  "==================")
p_inlet = 1.0
p_outlet = 0.0
dP_conv = 1e-3 * (p_inlet-p_outlet)  # convergence is achieved, when the relative pressure change is below this convergence criteria
print(f"Imposed pressure drop: {p_inlet - p_outlet} Pa")
# initial guess: linear
p = np.linspace(p_inlet*0.5, p_outlet, shape[0], endpoint=True)
p = np.tile(p.reshape(-1, 1, 1), reps=[1, shape[1], shape[2]]).reshape(-1, 1)
p_lin = p.copy()
p[pore_inlet] = p_inlet
p[pore_outlet] = p_outlet
c0, c1 = conns[:, 0], conns[:, 1]
print("\nStarting Jacobi iterations:")
print(  "============================")
print("it\tmax pressure change")
for i in range(1, n_iter+1):
    p_old = p.copy()
    for idp in range(p.size):
        if idp in pore_inlet or idp in pore_outlet:
            continue
        
        t_0 = np.where(c1 == idp)[0]
        c0_loc = c0[t_0]
        t_1 = np.where(c0 == idp)[0]
        c1_loc = c1[t_1]
        p[idp] = np.sum(g[t_0]*p_old[c0_loc]) + np.sum(g[t_1]*p_old[c1_loc])
        p[idp] /= np.sum(g[t_0]) + np.sum(g[t_1])
    dp_max = np.sqrt(np.max((p - p_old)**2))
    if (i % n_out) == 0 or i == (n_iter) or (dp_max < dP_conv):
        print(f"{i}\t{dp_max:1.2e}")
    if dp_max < dP_conv:
        break


In [ ]:
# Analysis and plotting
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from IPython.display import HTML
try:
    import plotly
except ImportError:
    import piplite
    await piplite.install("plotly")
    import plotly
from plotly import graph_objects as go



# compute total flow rate at outlet layer and permeability
print("Computing total flow rate at outlet... ", end="")
Q_tot = 0
for idp in pore_outlet:
    conn_0 = np.nonzero(conns[:, 0] == idp)[0]
    conn_1 = np.nonzero(conns[:, 1] == idp)[0]
    cid = -1
    for conn_id in conn_0:
        if conns[conn_id, 1] in pore_outlet:
            continue
        cid = conn_id
    if cid > -1:
        Q_tot += g[c_id] * (p[conns[cid, 1]] - p[idp])
        continue
    for conn_id in conn_1:
        if conns[conn_id, 0] in pore_outlet:
            continue
        cid = conn_id
    if cid == -1:
        raise ValueError('could not find an appropriate throat')
    Q_tot += g[cid] * (p[conns[cid, 0]] - p[idp])
print("done")
print(f"Total flow rate: {Q_tot} m^3/s")

K = Q_tot * mu * sz[0]/((p_inlet-p_outlet)*sz[1]*sz[2])
print(f'Permeability: {K} m^2')

# All flow rates
Q = np.abs(g* (p[conns[:, 1]]-p[conns[:, 0]])).reshape(-1)

mask_x = coords[conns[:, 1], 0] != coords[conns[:, 0], 0]
Q_min_x = Q[mask_x].min()
Q_max_x = Q[mask_x].max()

norm = Normalize(vmin=Q_min_x, vmax=Q_max_x)

# scale line thickness
linewidths = 1 + 8 * norm(Q)
linewidths[linewidths<0] = 0

cmap = plt.get_cmap("viridis")
colormap = cmap(norm(Q))

dP = p - p_lin
# norm = Normalize(vmin=dP.min(), vmax=dP.max())
# ax.scatter(*coords.T, s=100, c=p, cmap="viridis")

# ax.set_box_aspect(shape)

fig = go.Figure()

for n, ((i, j), value) in enumerate(zip(conns, Q)):
    color = colormap[n]

    fig.add_trace(
        go.Scatter3d(
            x=[coords[i,0], coords[j,0]],
            y=[coords[i,1], coords[j,1]],
            z=[coords[i,2], coords[j,2]],
            mode="lines",
            line=dict(
                color=color,
                width=linewidths[n]
            ),
            showlegend=False
        )
    )
fig.add_trace(
    go.Scatter3d(
        x=coords[:,0],
        y=coords[:,1],
        z=coords[:,2],
        mode="markers",
        marker=dict(
            size=6,
            color=dP,
            colorscale="Viridis",
            opacity=1.0
        )
    )
)
HTML(fig.to_html())
